**Dataset de esta edicion:** SQuAD 1.1 dev, 32 pasajes originales de Wikipedia y 16 preguntas redactadas por personas.

**Preparacion obligatoria** (una sola vez, desde la raiz de CC-0F4):

```bash
python -m Semana9.prepare_real_data
```

No se reemplaza por datos sinteticos si la descarga falla.

### **Cuaderno 9 - De workflows reproducibles a un agente con herramientas**
**CC-0F4 | Semana 9, 28 y 29 de octubre de 2026 | Teoría y demostración guiada**

En este cuaderno la unidad de análisis es la **decisión de control**. Primero reproduciremos un workflow fijo, luego una política adaptativa con reglas y finalmente un ciclo de herramientas al que se puede conectar un LLM. Cada etapa conserva contratos y permite comprobar resultados. El experimento offline no se confunde con una ejecución real de modelo.

**Límite curricular:** solo utilizamos contadores y observaciones transitorias de una ejecución. No se implementan working memory, episodic memory, semantic memory ni persistencia entre sesiones.

#### **1. Qué cambia respecto de las Semanas 6 y 7**
En Semana 6 aprendimos a validar una sola llamada a una herramienta. En Semana 7 comparamos componentes de recuperación y RAG sobre datos fijos. Aquí una aplicación debe elegir entre **buscar, leer, terminar o rechazar una acción**. El código conserva la autoridad para ejecutar. El criterio de terminación forma parte del algoritmo, no del diseño gráfico.

Preguntas iniciales: ¿qué acciones son legales?, ¿cuándo es útil llamar nuevamente al retriever?, ¿cómo se demuestra que un agente no entró en un bucle?

In [ ]:
from pathlib import Path
import json, os, sys
root = Path.cwd().resolve()
for directory in (root, *root.parents):
    if (directory / "Semana9/agent_benchmark.py").is_file():
        root = directory
        break
else:
    raise FileNotFoundError("Ejecuta desde la raiz de CC-0F4 con Semana9 disponible")
sys.path.insert(0, str(root))
from Semana9.agent_benchmark import (BM25Tool, jsonl, lexical_coverage,
                                     run_workflow, run_adaptive, evaluate_run)
from Semana9.agent_loop import (Limits, RestrictedTools, ScriptedPolicy, run_agent)
from Semana9.agent_demo import demo
print("Repositorio:", root)
from Semana9.agent_benchmark import real_paths
real_paths(root)  # Exige datos SQuAD reales antes de usar el laboratorio.


#### **2. Datos heredados y reproducibilidad**
La muestra SQuAD 1.1 tiene 8 articulos reales, 32 pasajes y 16 preguntas humanas con qrels a nivel de pasaje. No se redefine relevancia en esta semana. La herramienta de control es un BM25 local para todos los baselines. No afirmamos que sea el E5/FAISS o el híbrido de Semana 7.

In [ ]:
queries = jsonl(root / "Semana9/datos_reales/queries.jsonl")
corpus = jsonl(root / "Semana9/datos_reales/corpus.jsonl")
qrels = json.loads((root / "Semana9/datos_reales/qrels.json").read_text(encoding="utf-8"))
assert len(queries) == 16 and len(corpus) == 8
assert len({p["passage_id"] for d in corpus for p in d["passages"]}) == 32
print("Consultas:", len(queries), "Documentos:", len(corpus), "Qrels:", len(qrels))
print("Primer caso:", queries[0]["query_id"], queries[0]["query"])

#### **3. Arquitecturas: pipeline, workflow con ramas, control adaptativo y agente**
La frontera conceptual puede localizarse en el origen de las decisiones. Un workflow fijo contiene una secuencia de herramientas predeterminada. Un workflow con ramas puede observar y cambiar de trayectoria, pero lo hace conforme a reglas de código. Un agente LLM solicita una acción que el controlador valida.

No todos los workflows son lineales, ni todo bucle `while` convierte a un programa en agente.

In [ ]:
def workflow_chaining(query: str) -> dict:
    """Dos etapas predeterminadas, ejemplo de prompt chaining sin llamadas a LLM."""
    normalized = " ".join(query.casefold().split())
    return {"normalized_query": normalized,
            "validation": "accepted" if normalized else "rejected"}

def workflow_routing(query: str) -> str:
    """Ruta por reglas para estudiar el patron, no un clasificador LLM."""
    q = query.casefold()
    if "vpn" in q: return "redes/vpn"
    if "git" in q: return "repositorios"
    return "indice_general"

print(workflow_chaining(queries[0]["query"]))
print("Ruta:", workflow_routing(queries[0]["query"]))

#### **4. Cinco patrones de workflow**
- **Prompt chaining:** una salida validada alimenta la siguiente etapa.
- **Routing:** una categoría de entrada determina la ruta.
- **Parallelization:** divide trabajos independientes o contrasta múltiples intentos.
- **Evaluator-optimizer:** revisa una salida hasta aceptarla o alcanzar un límite.
- **Orchestrator-workers:** asigna subtareas dinámicamente, aquí solo se examina el patrón, no se construye un sistema multiagente (Semana 11).

Los siguientes ejemplos son algoritmos de control simples, no invocaciones ocultas a un modelo. Se estudia la topología antes de agregar inferencia probabilística.

In [ ]:
def evaluator_optimizer(initial: str, max_revisions: int = 2):
    """Ciclo acotado de validacion, util para explicar un workflow."""
    candidate = initial
    trace = []
    for i in range(max_revisions + 1):
        valid = len(candidate.split()) >= 4
        trace.append({"iteration": i, "valid": valid, "candidate": candidate})
        if valid:
            break
        if i < max_revisions:
            candidate += " con evidencia verificable"
    return trace

for step in evaluator_optimizer("Diagnostico DNS"):
    print(step)

#### **5. BM25 como herramienta común**
La implementación mantiene fijos corpus, tokenización, BM25 y ordenación. La política de control no observa las etiquetas qrels durante la ejecución. La política puede inspeccionar texto recuperado y decidir si continuar; los qrels solo se utilizan **después** para evaluar.

In [ ]:
tool = BM25Tool(corpus)
query = queries[0]["query"]
rank = tool.search(query, 3)
for row in rank:
    print(row["passage_id"], "score=", row["score"], "|", row["text"][:95])

#### **6. A0 y A1: workflows fijos**
A0 recupera `top_k=1` y A1 recupera `top_k=3`. Ambos tienen exactamente una llamada. A0 permite medir contexto mínimo y A1 cobertura de un contexto más amplio. Ninguna condición selecciona dinámicamente una acción.

In [ ]:
a0 = run_workflow(query, tool, top_k=1)
a1 = run_workflow(query, tool, top_k=3)
for r in (a0, a1):
    print(r["policy"], "citas", r["citations"], "llamadas", r["tool_calls"])
assert set(a0["citations"]).issubset(a1["citations"])

#### **7. B: controlador heurístico adaptativo**
B primero recupera un pasaje. Calcula la cobertura léxica entre la consulta y el pasaje; si queda bajo el umbral predeclarado 0.45, amplía a tres. Esta regla no usa un LLM ni qrels. La pregunta no es si B tiene más cobertura que A1: con ranking fijo **no puede**. La pregunta es si el costo adicional de las decisiones evita transportar contexto innecesario.

In [ ]:
b = run_adaptive(query, tool, threshold=0.45)
print(json.dumps({k: v for k, v in b.items() if k != "answer_excerpt"}, ensure_ascii=False, indent=2))
assert set(a0["citations"]).issubset(b["citations"])
assert set(b["citations"]).issubset(a1["citations"])

#### **8. Benchmark pareado y métrica operativa**
El indicador `evidence_hit` comprueba si una de las citas corresponde a un pasaje relevante. No evalúa verdad de la respuesta, fundamentación de todos los claims ni éxito de agente LLM. El proxy de contexto cuenta palabras según un tokenizador léxico, no tokens reales del LLM. La latencia es tiempo local del script BM25.

In [ ]:
summary = evaluate_run(root, root / "Semana9/resultados", threshold=0.45)
print("Datos SHA-256:", json.dumps(summary["data_sha256"], indent=2))
for policy, metrics in summary["results"].items():
    print(policy, json.dumps(metrics, ensure_ascii=False))

#### **9. Invariantes del resultado**

En el ranking fijo y sobre los mismos qrels, `Hit@A0 <= Hit@B <= Hit@A1`. No se debe interpretar el uso de dos búsquedas en B como una mejora algorítmica del retriever. El contexto puede ser más pequeño o no y la sobrecarga de llamadas debe analizarse con el conjunto completo.

In [ ]:
rows = jsonl(root / "Semana9/resultados/traces.jsonl")
by_q = {}
for row in rows:
    by_q.setdefault(row["query_id"], {})[row["policy"]] = row
assert len(by_q) == 16
for qid, policies in by_q.items():
    set0 = set(policies["workflow_fixed_top1"]["citations"])
    set1 = set(policies["workflow_fixed_top3"]["citations"])
    setb = set(policies["adaptive_rule_based"]["citations"])
    assert set0 <= setb <= set1
print("Monotonicidad de evidencia verificada en", len(by_q), "consultas")

#### **10. Diseño del controlador del agente**
El controlador implementa un conjunto de herramientas autorizadas: `search_documents`, `read_passage` y `finish`. La finalización es una decisión validada, no una herramienta externa. El sistema impone `max_steps`, `max_tool_calls`, rechazo por argumentos inválidos, deadline cooperativo y detección de firma de acción repetida.

El estado solo dura la ejecución actual: contadores, observaciones y citas recuperadas. Esta estructura **no implementa memoria de Semana 10**.

In [ ]:
restricted = RestrictedTools(tool)
first_id = tool.search(query, 3)[0]["passage_id"]
scripted = ScriptedPolicy([
    {"action": "search_documents", "arguments": {"query": query, "top_k": 3}},
    {"action": "read_passage", "arguments": {"passage_id": first_id}},
    {"action": "finish", "arguments": {
        "answer": "Contrastar la incidencia con el pasaje institucional recuperado.",
        "citations": [first_id]}}
])
replay = run_agent(query, scripted, restricted, Limits(max_steps=6, max_tool_calls=3))
print("Parada:", replay["stop_reason"], "| herramientas:", replay["tool_calls"])
for event in replay["trace"]:
    print(event["step"], event.get("action"), event["status"])
assert replay["stop_reason"] == "completed_with_citations"

#### **11. Atención: el replay anterior NO fue una ejecución LLM**
`ScriptedPolicy` contiene acciones escritas por el programador. Demuestra que el controlador valida contratos, recopila observaciones y permite terminar; **no demuestra autonomía del modelo**. En la condición real, `LocalTransformersPolicy` solicitará al modelo una decisión JSON a partir de la tarea y las últimas observaciones. El programa conservará validación y parada. Un modelo compacto puede devolver respuestas inválidas: esa posibilidad es un resultado técnico, no un defecto que deba ocultarse.

In [ ]:
run_real_llm = os.getenv("CC0F4_RUN_REAL_LLM", "0") == "1"
if run_real_llm:
    from Semana9.agent_policy import LocalTransformersPolicy
    real_policy = LocalTransformersPolicy(model_id=os.getenv(
        "CC0F4_AGENT_MODEL", "Qwen/Qwen2.5-0.5B-Instruct"))
    real = run_agent(query, real_policy, RestrictedTools(tool),
                     Limits(max_steps=6, max_tool_calls=3, deadline_seconds=180))
    print("LLM real, parada:", real["stop_reason"], "pasos:", real["step"])
    print("Tokens contados:", real["prompt_tokens"], real["generated_tokens"])
    print("Trazas:", json.dumps(real["trace"], ensure_ascii=False, indent=2)[:5000])
else:
    print("No se ejecuto LLM. Active CC0F4_RUN_REAL_LLM=1 para modo real voluntario.")

#### **12. Demostración de política y permisos**
Una herramienta solo puede leer un pasaje si antes fue recuperado. Una respuesta `finish` solo puede citar pasajes observados y no vacíos. Intentos de ejecutar comandos del sistema no están en la allowlist. Registramos el rechazo sin ejecutar la petición.

In [ ]:
bad = ScriptedPolicy([
    {"action": "run_shell", "arguments": {"command": "echo prohibido"}},
    {"action": "run_shell", "arguments": {"command": "echo prohibido"}},
])
security_demo = run_agent(query, bad, restricted)
print(security_demo["stop_reason"], "herramientas ejecutadas:", security_demo["tool_calls"])
assert security_demo["tool_calls"] == 0
assert security_demo["stop_reason"] == "invalid_action_limit"

#### **13. Planificación y replanning: qué cambia realmente**
Un plan docente puede ser: (1) buscar, (2) verificar un pasaje, (3) citar y finalizar. Una falla al leer un identificador inexistente debe activar una nueva decisión o una parada. El controlador no acepta el plan como permiso universal y valida cada paso. La siguiente celda muestra un intento inválido seguido de una alternativa válida. No hay persistencia de experiencias entre sesiones.

In [ ]:
recover = ScriptedPolicy([
    {"action": "read_passage", "arguments": {"passage_id": "D99-P99"}},
    {"action": "search_documents", "arguments": {"query": query, "top_k": 1}},
    {"action": "finish", "arguments": {"answer": "Consultar la fuente encontrada.",
                                      "citations": [first_id]}},
])
# first_id coincide con el primer pasaje de la misma busqueda.
recovery = run_agent(query, recover, restricted)
print(recovery["stop_reason"], "errores:", recovery["invalid_actions"])
assert recovery["invalid_actions"] == 1
assert recovery["stop_reason"] == "completed_with_citations"

#### **14. Límite de pasos y detección de ciclos**
Una firma de acción incluye nombre y argumentos. Si se repite exactamente después de no haber progresado, nuestro controlador docente termina. Esto es una heurística intencionalmente simple, no identifica equivalencias semánticas entre dos consultas reformuladas.

In [ ]:
repeat = ScriptedPolicy([
    {"action": "search_documents", "arguments": {"query": query, "top_k": 1}},
    {"action": "search_documents", "arguments": {"query": query, "top_k": 1}},
])
rpt = run_agent(query, repeat, restricted)
print("Motivo:", rpt["stop_reason"], "llamadas:", rpt["tool_calls"])
assert rpt["stop_reason"] == "repeated_action"
assert rpt["tool_calls"] == 1

#### **15. Qué significa una salida válida**
El controlador solo comprueba que `finish` tenga texto y citas observadas. No comprueba automáticamente la fidelidad de todas las afirmaciones, por lo que la etiqueta `completed_with_citations` es operacional. Para afirmar calidad semántica hacen falta criterios y anotaciones adicionales. Compare este resultado con las evaluaciones de RAG estudiadas en Semana 7.

#### **16. Preguntas adicionales**
1. Muestre la línea de código donde se impone la allowlist. ¿Por qué el prompt del modelo no basta?
2. ¿Por qué en un ranking inmutable A1 tiene al menos la cobertura de B?
3. Señale un caso en que B ahorre contexto, pero aumente llamadas.
4. Distinga `max_steps`, `max_tool_calls`, deadline y budget de tokens.
5. Explique por qué la simulación con ScriptedPolicy no prueba ReAct real.
6. ¿Qué necesitaría el experimento para comparar exactitud de respuesta y no solo cobertura de citas?
7. ¿Qué estado se conserva durante el bucle y qué contenidos corresponden a la Semana 10?.

**Lectura asociada:** `Lectura9-CC-0F4.md`, especialmente secciones 5 a 10.